In [ ]:
import numpy as np
import pandas as pd

# 1. Path to your downloaded 68 MB Excel File
file_path = './Data/Document from singhlink4.xlsx'

print('Loading all 9 sheets from Excel file... (this may take ~15-20 seconds)')
all_sheets = pd.read_excel(file_path, sheet_name=None, engine='openpyxl')
print('Loaded Sheets:', list(all_sheets.keys()))

# 2. Extract DataFrames
orders = all_sheets['orders']
items = all_sheets['order_items']
payments = all_sheets['order_payments']
reviews = all_sheets['order_reviews']
customers = all_sheets['customers']
products = all_sheets['products']
sellers = all_sheets['sellers']
geolocation = all_sheets['geolocation']
category_translation = all_sheets['category_translation']

# 3. Clean Geolocation (Pre-aggregate to prevent row duplication)
geo_clean = (
    geolocation.groupby('geolocation_zip_code_prefix')
    .agg(
        lat=('geolocation_lat', 'mean'),
        lng=('geolocation_lng', 'mean'),
        city=('geolocation_city', 'first'),
        state=('geolocation_state', 'first'),
    )
    .reset_index()
)

# 4. Add English Product Category Names
products_tr = products.merge(
    category_translation, on='product_category_name', how='left'
)
products_tr['product_category_name_english'] = products_tr[
    'product_category_name_english'
].fillna('Other / Uncategorized')

# 5. Filter Delivered Orders & Calculate Delivery Metrics
delivered_orders = orders[orders['order_status'] == 'delivered'].copy()

date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
]
for col in date_cols:
  delivered_orders[col] = pd.to_datetime(delivered_orders[col])

# Key Metrics
delivered_orders['delivery_delay_days'] = (
    delivered_orders['order_delivered_customer_date']
    - delivered_orders['order_estimated_delivery_date']
).dt.total_seconds() / (24 * 3600)

delivered_orders['actual_delivery_time_days'] = (
    delivered_orders['order_delivered_customer_date']
    - delivered_orders['order_purchase_timestamp']
).dt.total_seconds() / (24 * 3600)

delivered_orders['is_late'] = (
    delivered_orders['delivery_delay_days'] > 0
).astype(int)
delivered_orders['year_month'] = delivered_orders[
    'order_purchase_timestamp'
].dt.to_period('M')

# 6. Clean Reviews (Deduplicate)
reviews_clean = reviews.sort_values('review_answer_timestamp').drop_duplicates(
    subset='order_id', keep='last'
)

# 7. Aggregate Line Items & Payments per Order
order_items_agg = (
    items.groupby('order_id')
    .agg(
        total_items=('order_item_id', 'count'),
        total_price=('price', 'sum'),
        total_freight=('freight_value', 'sum'),
        primary_seller_id=('seller_id', 'first'),
        primary_product_id=('product_id', 'first'),
    )
    .reset_index()
)

payments_agg = (
    payments.groupby('order_id')
    .agg(
        total_payment_value=('payment_value', 'sum'),
        primary_payment_type=('payment_type', 'first'),
        max_installments=('payment_installments', 'max'),
    )
    .reset_index()
)

# 8. Create Final Master Table
df_master = (
    delivered_orders.merge(customers, on='customer_id', how='left')
    .merge(order_items_agg, on='order_id', how='left')
    .merge(payments_agg, on='order_id', how='left')
    .merge(
        reviews_clean[['order_id', 'review_score', 'review_comment_message']],
        on='order_id',
        how='left',
    )
    .merge(
        products_tr[
            ['product_id', 'product_category_name_english', 'product_weight_g']
        ],
        left_on='primary_product_id',
        right_on='product_id',
        how='left',
    )
    .merge(sellers, left_on='primary_seller_id', right_on='seller_id', how='left')
)
print('Total Rows in Master Dataset:', df_master.shape[0])

Loading all 9 sheets from Excel file... (this may take ~15-20 seconds)


In [ ]:
df = df_master.copy()

In [38]:
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay_days,actual_delivery_time_days,...,max_installments,review_score,review_comment_message,product_id,product_category_name_english,product_weight_g,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,-7.107488,8.436574,...,1.0,4,"Não testei o produto ainda, mas ele veio corre...",87285b34884572647811a353c7ac498a,housewares,500.0,3504c0cb71d7fa48d967e0e4c94d59d9,9350,maua,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,-5.355729,13.782037,...,1.0,4,Muito bom o produto.,595fac2a385ac33a80bd5114aec74eb8,perfumery,400.0,289cdb325fb7e7f891c38608bf9e0962,31570,belo horizonte,SP
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,-17.245498,9.394213,...,3.0,5,NaN,aa4383b373c6aca5d8797843e5594415,auto,420.0,4869f7a5dfa277a7dca6462dcf3b52b2,14840,guariba,SP
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,-12.980069,13.208750,...,1.0,5,O produto foi exatamente o que eu esperava e e...,d0b61bfb1de832b15ba9d266ca96e5b0,pet_shop,450.0,66922902710d126a0e7d26b0e3805106,31842,belo horizonte,MG
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,-9.238171,2.873877,...,1.0,5,NaN,65266b2da20d04dbe00c5c2d3bb7859e,stationery,250.0,2c9e548be18521d1c43cde1c582c6de8,8752,mogi das cruzes,SP


In [39]:
df.shape

(96478, 33)

In [40]:
df.duplicated().sum()

np.int64(0)

In [41]:
df.isnull().sum()

,0
order_id,0
customer_id,0
order_status,0
order_purchase_timestamp,0
order_approved_at,14
order_delivered_carrier_date,2
order_delivered_customer_date,8
order_estimated_delivery_date,0
delivery_delay_days,8
actual_delivery_time_days,8


In [42]:
df['order_approved_at'] = df['order_approved_at'].fillna(
    df['order_purchase_timestamp']
)

In [43]:
print(str(round(df['review_comment_message'].isnull().sum() * 100 / df.shape[0], 2)) + '%')

59.01%


In [44]:
df['review_comment_message'].sample(5)

,review_comment_message
18289,NaN
39001,Primeira experiência boa.
49417,NaN
38822,NaN
96377,NaN


In [45]:
df['review_comment_message'] = df[
    'review_comment_message'
].fillna('No Review Comment')

In [46]:
df['product_weight_g'] = df['product_weight_g'].fillna(
    df.groupby('product_category_name_english')[
        'product_weight_g'
    ].transform('median')
)
print(
    'Null values in product_weight_g after fix:',
    df['product_weight_g'].isnull().sum(),
)

Null values in product_weight_g after fix: 0


In [47]:
df.dropna(inplace = True)

In [48]:
print(
    'Shape of DF after Cleaning \n',
    'Rows = ', df.shape[0],'\n',
    'Columns = ', df.shape[1],
)

Shape of DF after Cleaning 
 Rows =  96468 
 Columns =  33


In [49]:
df.isnull().sum().values > 0

array([False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False])

In [ ]:
df.to_csv('./Cleaned_Data/master_olist_cleaned_final.csv', index = False)
print('SUCCESS! Master file saved as master_olist_cleaned_final.csv')

SUCCESS! Master file saved as master_olist_cleaned_final.csv
